# 14 — SQL Filter + Text Search on Real Recipe Data

`SQLFilter` (`simlar_engine.SQLFilter`) is a standalone SQL-driven metadata
pre-filter -- it owns its own SQLite connection and returns document
*positions*, not a post-hoc filter of an already-ranked top-k. A `WHERE`
clause decides what's in scope; `RelevanceIndex` (BM25) only ranks within it.

Same pattern as `examples/12_sql_filter_pipeline.py`, but against a real,
messy corpus instead of a six-row toy one: ~2,000 recipes sampled from the
["Extended Recipes Dataset: 64K Dishes"](https://www.kaggle.com/datasets/wafaaelhusseini/extended-recipes-dataset-64k-dishes)
(Kaggle) -- the same CSV used by the `pgvector_recipe_demo` project.

`RelevanceIndex.search()`/`search_raw()` take a `candidates=` kwarg directly
-- a single flat array of positions applied to that one query -- so **one**
`RelevanceIndex` is built over the whole corpus, and every filtered scenario
below just calls `.search(query, candidates=positions)` on it. No per-filter
re-indexing needed: SQLFilter decides what's in scope, the same full index
ranks within it.

In [1]:
import json

import pandas as pd
from simlar import RelevanceIndex
from simlar_engine import SQLFilter

CSV_PATH = "/home/aramirez/tekdatum/pgvector_recipe_demo/data/recipes_sample.csv"

## Load the data

`cuisine_list` / `course_list` are JSON-encoded list columns (a recipe can
carry several course tags -- `breakfast`, `dessert`, etc.); `total_min` is
derived once so it can be used directly as a filter column.

In [1]:
df = pd.read_csv(CSV_PATH)
df["cuisine_list"] = df["cuisine_list"].apply(json.loads)
df["course_list"] = df["course_list"].apply(json.loads)
df["total_min"] = df["est_prep_time_min"] + df["est_cook_time_min"]
print(f"Loaded {len(df)} recipes from {CSV_PATH}")
print(f"Columns used for filtering: is_vegan, is_gluten_free, difficulty, total_min, "
      f"healthiness_score, course_list")

Loaded 2026 recipes from /home/aramirez/tekdatum/pgvector_recipe_demo/data/recipes_sample.csv
Columns used for filtering: is_vegan, is_gluten_free, difficulty, total_min, healthiness_score, course_list


## Build the SQL filter

`SQLFilter.add(ids, metadata)` registers one row of metadata per document,
positions assigned by append order -- the same order `RelevanceIndex.add()`
is given below, so the two stay aligned without sharing a connection.
`course_list` is multi-valued, so it's flattened into boolean columns
(`is_breakfast`, `is_dessert`) the filter can actually query -- `SQLFilter`'s
metadata table is scalar columns, not arrays.

In [1]:
ids = [str(i) for i in df.index]
metadata = [
    {
        "is_vegan": int(row.is_vegan),
        "is_gluten_free": int(row.is_gluten_free),
        "difficulty": row.difficulty,
        "total_min": int(row.total_min),
        "healthiness_score": int(row.healthiness_score),
        "is_breakfast": int("breakfast" in row.course_list),
        "is_dessert": int("dessert" in row.course_list),
    }
    for row in df.itertuples()
]
filt = SQLFilter()
filt.add(ids, metadata)
print(f"SQLFilter built over {filt.size} recipes")

SQLFilter built over 2026 recipes


## Build one RelevanceIndex over the whole corpus

Built once, reused for every scenario below -- filtered or not.

In [1]:
full_index = RelevanceIndex()
full_index.add(ids=ids, texts=df["combined_text"].fillna("").tolist())
print(f"RelevanceIndex built over {full_index.size} recipes")


def show(results):
    for r in results:
        title = df.loc[int(r.id), "recipe_title"]
        print(f"  {r.score:6.3f}  {title}")

RelevanceIndex built over 2026 recipes


## Baseline: unrestricted search

`candidates=None` (the default) -- every recipe is a candidate.

In [1]:
query = "butter chicken"
print(f'Query: "{query}"\n')
show(full_index.search(query, k=5))

Query: "butter chicken"

   2.034  Buffalo Chicken Wraps
   1.990  Easy Slow Cooker Chicken and Dumplings
   1.958  Chicken Marsala Over White Rice
   1.942  Chef John's Homemade Chicken Noodle Soup
   1.931  Chicken with Lemon-Caper Sauce


## Filter 1 — "Vegan dinners similar to butter chicken"

`WHERE is_vegan = 1`, 377 of 2,026 recipes qualify. `filt.where(...).positions()`
returns the matching positions; `full_index.search(query, candidates=positions)`
reuses the same index, just restricted to them.

**Worth being honest about**: the top hit, "Old-Fashioned Homemade Hard
Candy," is vegan but nothing like butter chicken -- the same failure mode the
`pgvector_recipe_demo` notebook's D1 example hits with vector similarity
(`is_vegan` legitimately removes every real chicken dish, and BM25 then has
no real lexical overlap left to rank on, hence the `0.000` scores below it).
The filter did its job correctly; there just isn't a good answer left inside
it for this query.

In [1]:
print("-- SQL filter: is_vegan = 1 --")
vegan_positions = filt.where("is_vegan", "=", 1).positions()
print(f"({len(vegan_positions)} of {len(df)} recipes qualify)\n")
show(full_index.search(query, k=5, candidates=vegan_positions))

-- SQL filter: is_vegan = 1 --
(377 of 2026 recipes qualify)

   0.220  Old-Fashioned Homemade Hard Candy
   0.000  Homemade Baked Beans
   0.000  Easter Meat Pie
   0.000  Hungarian Pizzelle
   0.000  Peanut Butter French Toast


## Filter 2 — "Easy gluten-free desserts, chocolate cake style"

`WHERE is_gluten_free = 1 AND is_dessert = 1 AND difficulty = 'easy'` --
three AND'ed conditions, 29 of 2,026 recipes qualify. Every condition's
value is bound as a parameter; `column`/`op` are the only pieces spliced
into SQL text, and both are validated (identifier check, fixed operator
whitelist) before that happens.

In [1]:
query2 = "chocolate cake"
print(f'Query: "{query2}"\n')
print("-- SQL filter: is_gluten_free = 1 AND is_dessert = 1 AND difficulty = 'easy' --")
easy_gf_dessert = (
    filt.where("is_gluten_free", "=", 1)
    .where("is_dessert", "=", 1)
    .where("difficulty", "=", "easy")
    .positions()
)
print(f"({len(easy_gf_dessert)} of {len(df)} recipes qualify)\n")
show(full_index.search(query2, k=5, candidates=easy_gf_dessert))

Query: "chocolate cake"

-- SQL filter: is_gluten_free = 1 AND is_dessert = 1 AND difficulty = 'easy' --
(29 of 2026 recipes qualify)

   3.413  Delicious Chocolate Cornflake Cakes
   2.114  Chocolate Balls
   2.114  Chocolate Ricotta Mousse
   2.103  Smooth Chocolate Icing
   2.098  Red Velvet Mug Cakes


## Filter 3 — "Healthy, quick comfort food"

`WHERE healthiness_score >= 70 AND total_min <= 30` -- numeric range filters
on a derived column (`total_min = est_prep_time_min + est_cook_time_min`),
87 of 2,026 recipes qualify.

In [1]:
query3 = "warm soup"
print(f'Query: "{query3}"\n')
print("-- SQL filter: healthiness_score >= 70 AND total_min <= 30 --")
healthy_quick = filt.where("healthiness_score", ">=", 70).where("total_min", "<=", 30).positions()
print(f"({len(healthy_quick)} of {len(df)} recipes qualify)\n")
show(full_index.search(query3, k=5, candidates=healthy_quick))

Query: "warm soup"

-- SQL filter: healthiness_score >= 70 AND total_min <= 30 --
(87 of 2026 recipes qualify)

   1.704  Simple Strawberry Soup
   1.000  California Club Chicken Wraps
   0.000  Easter Meat Pie
   0.000  Homemade Baked Beans
   0.000  Hungarian Pizzelle


## Takeaway

`SQLFilter` guarantees what must be true (diet flags, difficulty, numeric
thresholds, multi-valued tags flattened to booleans) as a real SQL `WHERE`
clause -- no raw SQL text ever comes from the caller, values are always bound
parameters. `RelevanceIndex` (BM25) only ranks *within* whatever the filter
lets through, and it's the *same* index every time: `candidates=` masks
scores at query time rather than requiring a fresh index per filter, which
also means BM25's term statistics (IDF) stay computed over the whole corpus
instead of shifting with each filter's subset -- the correct behavior, and
the reason results here differ slightly from a naive "reindex the subset"
approach. When the filter removes every genuinely relevant document (see
Filter 1), that's a correct, honest outcome, not a search bug -- there's
nothing left worth ranking highly, so the scores collapse toward `0.000`.